### Imports

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.transforms import InterpolationMode
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
from PIL import Image

### Helper Functions

In [ ]:
@torch.inference_mode()
def collect_predictions(model, dataloader, device):
    """  
    Collects predictions from the model for the entire dataset in the dataloader.
    """
    model.eval()
    model.to(device)

    all_logits = []
    all_y = []

    for x, y in dataloader:
        x = x.to(device)
        logits = model(x)              
        all_logits.append(logits.cpu())
        all_y.append(y.cpu())

    logits = torch.cat(all_logits, dim=0)       
    y_true = torch.cat(all_y, dim=0).numpy()    
    probs = torch.softmax(logits, dim=1).numpy()
    y_pred = probs.argmax(axis=1)               

    return y_true, y_pred, probs, logits.numpy()


In [ ]:
def plot_confusion_matrix(cm, class_names, normalize=True):
    """
    Plots the confusion matrix using matplotlib. 
    If normalize is True, the confusion matrix is normalized to show percentages instead of raw counts.
    """
    cm_display = cm.astype(np.float64)
    if normalize:
        cm_display = cm_display / cm_display.sum(axis=1, keepdims=True).clip(min=1)

    plt.figure(figsize=(8, 7))
    plt.imshow(cm_display)
    plt.xticks(range(len(class_names)), class_names, rotation=45, ha="right")
    plt.yticks(range(len(class_names)), class_names)
    plt.colorbar()
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix" + (" (Normalized)" if normalize else ""))
    plt.tight_layout()
    plt.show()

In [ ]:
def show_misclassified(mis_df, n=12):
    """
    Displays a grid of misclassified images along with their true labels, predicted labels, and confidence scores.
    """
    n = min(n, len(mis_df))
    plt.figure(figsize=(12, 8))
    for j in range(n):
        row = mis_df.iloc[j]
        img = Image.open(row["path"]).convert("RGB")

        plt.subplot(3, 4, j+1)
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"T:{row['true_label']}\nP:{row['pred_label']} ({row['conf']:.2f})")
    plt.tight_layout()
    plt.show()

### Model

In [ ]:
class PlantDataModule(pl.LightningDataModule):
    def __init__(self, train_dir, val_dir, test_dir, batch_size=32, num_workers=4):
        """
        Initializes the PlantDataModule with directories for training, validation, and testing datasets, along with batch size and number of workers for data loading.
        """
        super().__init__()
        self.train_dir = train_dir
        self.val_dir = val_dir
        self.test_dir = test_dir
        self.batch_size = batch_size
        self.num_workers = num_workers

        # Use the default weights for EfficientNet_B0 to get the correct normalization values
        weights = EfficientNet_B0_Weights.DEFAULT
        self.imagenet_mean = weights.transforms().mean
        self.imagenet_std = weights.transforms().std

        # Define data augmentation and preprocessing transforms for training and evaluation
        self.train_transform = transforms.Compose([
            transforms.Resize(256, interpolation=InterpolationMode.BILINEAR),
            transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
            transforms.ToTensor(),
            transforms.Normalize(mean=self.imagenet_mean, std=self.imagenet_std),
            transforms.RandomErasing(p=0.25),
        ])

        # For validation and testing, we only apply resizing, center cropping, and normalization without augmentation
        self.eval_transform = transforms.Compose([
            transforms.Resize(256, interpolation=InterpolationMode.BILINEAR),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=self.imagenet_mean, std=self.imagenet_std),
        ])

    def setup(self, stage=None):
        """  
        Initializes the datasets for training, validation, and testing using ImageFolder and applies the appropriate transforms.
        """
        self.train_dataset = ImageFolder(root=self.train_dir, transform=self.train_transform)
        self.val_dataset = ImageFolder(root=self.val_dir, transform=self.eval_transform)
        self.test_dataset = ImageFolder(root=self.test_dir, transform=self.eval_transform)

    def train_dataloader(self):
        """  
        Returns the DataLoader for the training dataset with shuffling enabled and appropriate settings for performance.
        """
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers, pin_memory=torch.cuda.is_available(), persistent_workers=(self.num_workers > 0))

    def val_dataloader(self):
        """  
        Returns the DataLoader for the validation dataset with shuffling disabled and appropriate settings for performance.
        """
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers, pin_memory=torch.cuda.is_available(), persistent_workers=(self.num_workers > 0))

    def test_dataloader(self):
        """
        Returns the DataLoader for the testing dataset with shuffling disabled and appropriate settings for performance.
        """
        return DataLoader(self.test_dataset, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers, pin_memory=torch.cuda.is_available(), persistent_workers=(self.num_workers > 0))

In [ ]:
class EfficientNetFineTuner(pl.LightningModule):
    def __init__(self, n_classes, lr_head=1e-3, lr_backbone=1e-4, weight_decay=1e-4):
        """
        A PyTorch Lightning module for fine-tuning an EfficientNet_B0 model on a custom plant classification task.
        """
        super().__init__()
        self.save_hyperparameters()

        # Load the EfficientNet_B0 model with pre-trained weights
        weights = EfficientNet_B0_Weights.DEFAULT
        self.model = efficientnet_b0(weights=weights)

        # Replace the final classification layer to match the number of classes in our dataset
        in_features = self.model.classifier[1].in_features
        self.model.classifier[1] = nn.Linear(in_features, n_classes)

        self.criterion = nn.CrossEntropyLoss()

    def forward(self, x):
        """
        Forward pass through the model to get logits for the input batch of images.
        """
        return self.model(x)
    
    def freeze_backbone(self):
        """
        Freeze the backbone (feature extractor) layers of the model to prevent them from being updated during training.
        """
        # Freeze all layers in the backbone
        for param in self.model.features.parameters():
            param.requires_grad = False

        # Unfreeze the classifier head to allow training of the new layers
        for param in self.model.classifier.parameters():
            param.requires_grad = True

    def unfreeze_last_n_blocks(self, n=3):
        """
        Unfreeze the last n blocks of the backbone to allow fine-tuning of these layers.
        """
        for param in self.model.features[-n:].parameters():
            param.requires_grad = True

    def training_step(self, batch, batch_idx):
        """
        Perform a training step by computing the loss and accuracy for the current batch.
        Metrics logged for monitoring.
        """
        x, y = batch

        # Forward pass to get logits and compute loss and accuracy
        logits = self(x)
        loss = self.criterion(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()

        # Logging training loss and accuracy for monitoring
        self.log('train_loss', loss, prog_bar=True, on_epoch=True)
        self.log('train_acc', acc, prog_bar=True, on_epoch=True)
        return loss
    
    def validation_step(self, batch, batch_idx):
        """
        Perform a validation step by computing the loss and accuracy for the current batch.
        Metrics logged for monitoring.
        """
        x, y = batch

        # Forward pass to get logits and compute loss and accuracy
        logits = self(x)
        loss = self.criterion(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()

    # Logging validation loss and accuracy for monitoring
        self.log('val_loss', loss, prog_bar=True, on_epoch=True)
        self.log('val_acc', acc, prog_bar=True, on_epoch=True)

    def test_step(self, batch, batch_idx):
        """ 
        Perform a test step by computing the loss and accuracy for the current batch. 
        Metrics logged for monitoring. 
        """
        x, y = batch

        # Forward pass to get logits and compute loss and accuracy
        logits = self(x)
        loss = self.criterion(logits, y)
        acc = (logits.argmax(dim=1) == y).float().mean()

        # Logging test loss and accuracy for monitoring
        self.log('test_loss', loss, prog_bar=True, on_epoch=True)
        self.log('test_acc', acc, prog_bar=True, on_epoch=True)

    def configure_optimizers(self):
        """
        Configure the optimizer with different learning rates for the backbone and head of the model.
        """
        backbone_params = [param for param in self.model.features.parameters() if param.requires_grad]
        head_params = [param for param in self.model.classifier.parameters() if param.requires_grad]

        optimizer = torch.optim.AdamW([
            {'params': backbone_params, 'lr': self.hparams.lr_backbone},
            {'params': head_params, 'lr': self.hparams.lr_head}
        ], weight_decay=self.hparams.weight_decay)
        return optimizer

In [ ]:
pl.seed_everything(42, workers=True)

# Initialize the data module with the appropriate directories for training, validation, and testing datasets
data_module = PlantDataModule(
    train_dir='plant_img_train',
    val_dir='plant_img_val',
    test_dir='plant_img_test'
)

# Initialize the model with the number of classes equal to the number of source directories (classes) in the dataset
model = EfficientNetFineTuner(n_classes=len(source_dirs))

# Set up callbacks for model checkpointing and early stopping based on validation accuracy during both phases of training
checkpoint_callback_1 = ModelCheckpoint(
    monitor='val_acc',
    mode='max',
    save_top_k=1,
    filename='efficientnet-phase1-finetuned-{epoch:02d}-{val_acc:.4f}'
)

early_stopping_callback_1 = EarlyStopping(
    monitor='val_acc',
    mode='max',
    patience=5,
)

checkpoint_callback_2 = ModelCheckpoint(
    monitor='val_acc',
    mode='max',
    save_top_k=1,
    filename='efficientnet-phase2-finetuned-{epoch:02d}-{val_acc:.4f}'
)

early_stopping_callback_2 = EarlyStopping(
    monitor='val_acc',
    mode='max',
    patience=5,
)

In [ ]:
model.freeze_backbone()

# Set up the trainer for the first phase of training with the appropriate callbacks and settings for performance
trainer_1 = pl.Trainer(
    max_epochs=5,
    accelerator='auto',
    devices='auto',
    precision='32',
    callbacks=[checkpoint_callback_1, early_stopping_callback_1],
    log_every_n_steps=10,
)

trainer_1.fit(model, datamodule=data_module)

In [ ]:
model.unfreeze_last_n_blocks(n=3)

# Set up the trainer for the second phase of training with the appropriate callbacks and settings for performance
trainer_2 = pl.Trainer(
    max_epochs=20,
    accelerator='auto',
    devices='auto',
    precision='32',
    callbacks=[checkpoint_callback_2, early_stopping_callback_2],
    log_every_n_steps=10,
)

trainer_2.fit(model, datamodule=data_module)

### Test Model

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

best_path = trainer_2.checkpoint_callback.best_model_path
model = EfficientNetFineTuner.load_from_checkpoint(best_path, n_classes=len(source_dirs))

test_loader = data_module.test_dataloader()
y_true, y_pred, probs, logits = collect_predictions(model, test_loader, device)

test_ds = data_module.test_dataset  
class_names = test_ds.classes       
file_paths = [p for (p, _) in test_ds.samples]


### Results Analysis

In [ ]:
cm = confusion_matrix(y_true, y_pred)
print("Classes:", class_names)
print("\nClassification report:\n")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

In [ ]:
plot_confusion_matrix(cm, class_names, normalize=True)

In [ ]:
top1_conf = probs.max(axis=1)
correct = (y_pred == y_true)

print("Mean confidence (correct):", top1_conf[correct].mean())
print("Mean confidence (wrong):  ", top1_conf[~correct].mean())
print("Num wrong:", (~correct).sum())

In [ ]:
show_misclassified(mis, n=12)